# Análisis gráfico de reacciones

## Objetivo
Analizar gráficamente las **propiedades fisicoquímicas** de 10 compuestos halogenados y las **reacciones** y **obtenciones** asociadas a ellos, usando como única fuente el archivo Excel `Proyecto de organica V12.xlsx`.

## Datos de partida
| Hoja del Excel | Contenido | Registros |
|---|---|---|
| **Compuestos** | Una fila por compuesto y una columna por propiedad: peso molecular, punto de fusión, punto de ebullición, densidad, pKa, momento dipolar, solubilidad en agua y presión de vapor | 10 compuestos |
| **Reacciones** | Reacciones en las que participa cada compuesto (tipo, medio, propiedades, etc.) | 100 (10 por compuesto) |
| **Obtenciones** | Reacciones con las que se obtiene cada compuesto | 100 (10 por compuesto) |

Algunas propiedades no tienen dato para todos los compuestos; esas celdas quedan vacías y **no se grafican** (no se inventan valores).

## Cómo ejecutar
1. Abrir este cuaderno en Google Colab (o Jupyter).
2. Subir a Colab el archivo `Proyecto de organica V12.xlsx`.
3. Ejecutar las celdas **en orden** (1 → 10) o usar *Ejecutar todo*.

## Contenido
| Celda | Proceso |
|---|---|
| 1 | Librerías y colores |
| 2 | Cargar y limpiar los datos |
| 3 | Funciones auxiliares |
| 4 | Gráficas de líneas dobles |
| 5 | Caja de solubilidad |
| 6 | Violín: Reacciones vs Obtenciones |
| 7 | Mapa de calor de correlación |
| 8 | Tipos de reacción (barras) |
| 9 | Lollipop: comparación con el Cloroformo |
| 10 | Tortas: medio de reacción |

## Celda 1 · Librerías y colores


**Proceso**
1. Se importan las librerías que usa todo el análisis:

| Librería | Para qué se usa |
|---|---|
| `os`, `re`, `textwrap`, `datetime` | Buscar el archivo, limpiar texto con expresiones regulares, partir nombres largos y detectar fechas mal leídas |
| `numpy` | Cálculos numéricos (percentiles, logaritmos, arreglos) |
| `pandas` | Leer el Excel y manejar las tablas (DataFrame) |
| `matplotlib` | Dibujar todas las gráficas |
| `scipy` | Ordenar el mapa de calor por agrupamiento jerárquico |

2. Se definen **todos los colores** de las gráficas juntos (azul, naranja, rojo, turquesa, beige…). Así, cambiar un color en esta celda lo cambia en todas las gráficas donde se usa.

In [ ]:
# ============================================================
# CELDA 1 · LIBRERÍAS Y COLORES
# ============================================================

# ---- Librerías ----
import os                                    # buscar el archivo Excel en las carpetas de Colab
import re                                    # expresiones regulares para limpiar texto
import textwrap                              # partir nombres largos en varias líneas
import datetime                              # detectar fechas que Excel confundió con números

import numpy as np                           # cálculos con números y arreglos
import pandas as pd                          # tablas (DataFrame) y lectura del Excel
import matplotlib.pyplot as plt              # todas las gráficas

from matplotlib.colors import LinearSegmentedColormap, to_hex    # degradado turquesa
from matplotlib.patches import FancyBboxPatch, Rectangle, Patch  # cuadros redondeados y leyendas
from matplotlib.lines import Line2D                              # leyenda del lollipop
from scipy.cluster.hierarchy import linkage, leaves_list         # ordenar el mapa de correlación

# ---- Colores (aquí se cambian para todas las gráficas) ----
AZUL = "#2f74c0"             # líneas dobles (eje izquierdo), lollipop y violín (Reacciones)
NARANJA = "#e8683a"          # líneas dobles (eje derecho), lollipop y violín (Obtenciones)
ROJO = "#d64545"             # barras de tipos de reacción: el dato más alto
AZUL_CLARO = "#a9cce3"       # barras de tipos de reacción: los demás datos
ROSA, VERDE = "#e8a0a0", "#c5e0a0"   # caja de solubilidad (mitad izquierda / mitad derecha)
FONDO = "white"              # fondo de las tortas

# Degradado turquesa del mapa de calor
turquesa = LinearSegmentedColormap.from_list(
    "turquesa", ["#c4efee", "#6fdada", "#2fb9bb", "#1f8c8f", "#0f3b3d"])
turquesa.set_bad("#e4e4e4")                  # celdas sin dato: gris clarito

# Colores de la torta de "Medio de reacción" (paleta crema/beige)
COLORES_MEDIO = {"Ácido": "#cfb384", "Básico": "#e0caa1", "Neutro": "#f6e3b9", "Ácido/Básico": "#ffefcb"}


## Celda 2 · Cargar y limpiar los datos


**Proceso**
1. **Buscar el archivo** `Proyecto de organica V12.xlsx` en las carpetas habituales (la del cuaderno y `/content`). Si no está, se muestra un error claro.
2. **Limpiar los valores** con la función `a_numero`. En el Excel hay celdas escritas como texto, y la función las convierte así:

| Formato en el Excel | Resultado |
|---|---|
| `5.061 x 10⁵` o `3,69 x 10-3` (notación científica, con superíndices) | número decimal |
| `58-59` (rango) | promedio de los dos extremos |
| `No aplica`, vacío, «verificar» | `NaN` (dato faltante) |
| coma decimal (`3,5`) | punto decimal (`3.5`) |

3. **Leer la hoja `Compuestos`** en `df` y aplicar `a_numero` a todas las propiedades.
4. **Localizar cada columna por palabra clave** (`col_peso`, `col_dens`, `col_fus`, `col_ebu`, `col_sol`), para no depender del nombre exacto del título.
5. **Leer las hojas `Reacciones` y `Obtenciones`** en `rx` y `ob`, descartando filas sin reactante o sin producto, y normalizando los espacios de los títulos.

**Resultado:** `df` queda con 10 compuestos y `rx` y `ob` con 100 registros cada una. Datos disponibles por propiedad en `df`: peso molecular 10, punto de ebullición 10, densidad 10, punto de fusión 9, momento dipolar 5, presión de vapor 5, solubilidad 4 y pKa 1. Las propiedades con pocos datos se grafican con cuidado (ver cada celda).

In [ ]:
# ============================================================
# CELDA 2 · CARGAR Y LIMPIAR LOS DATOS
# ============================================================

# ---- 1. Buscar el archivo Excel ----
for ruta in ["Proyecto de organica V12.xlsx",                       # misma carpeta del cuaderno
              "/Proyecto de organica V12.xlsx",
              "/content/Proyecto de organica V12.xlsx",
              "/content/Análisis_Gráfico_Reacciones/Proyecto de organica V12.xlsx"]:
    if os.path.exists(ruta):
        archivo = ruta
        break
else:
    raise FileNotFoundError("No encontré 'Proyecto de organica V12.xlsx'. Súbelo a Colab.")
print("Usando:", archivo)

# ---- 2. Función que convierte las celdas del Excel en números ----
SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻", "0123456789-")

def a_numero(x):
    """Convierte '5.061 x 10⁵', '3,69 x 10-3', '58-59', 'No aplica' en número (o NaN)."""
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    if x is None:
        return np.nan
    s = str(x).translate(SUP).strip().replace("\t", "").replace("\xa0", " ")
    if s == "" or s.lower().startswith("no aplica") or "verificar" in s.lower():
        return np.nan
    s = s.replace("×", "x").replace("−", "-")
    m = re.match(r"^([-+]?\d+(?:[.,]\d+)?)\s*[xX]\s*10\^?\s*([-+]?\d+)$", s)
    if m:  # notación científica
        return float(m.group(1).replace(",", ".")) * 10 ** int(m.group(2))
    m = re.match(r"^([-+]?\d+(?:[.,]\d+)?)\s*-\s*(\d+(?:[.,]\d+)?)$", s)
    if m:  # rango 58-59 -> promedio
        return (float(m.group(1).replace(",", ".")) + float(m.group(2).replace(",", "."))) / 2
    try:
        return float(s.replace(",", "."))
    except ValueError:
        return np.nan

# ---- 3. Hoja "Compuestos": una fila por compuesto, una columna por propiedad ----
df = pd.read_excel(archivo, sheet_name="Compuestos")
df.columns = [c.strip() for c in df.columns]
nombre = df.columns[0]                                  # columna con el nombre del compuesto
props = [c for c in df.columns if c != nombre]          # columnas de propiedades
for c in props:
    df[c] = df[c].apply(a_numero)                       # todo a número

# Nombre exacto de cada columna (se buscan por palabra clave)
col_dens = [c for c in props if "densidad" in c.lower()][0]
col_peso = [c for c in props if "peso" in c.lower()][0]
col_fus  = [c for c in props if "fusi" in c.lower()][0]
col_ebu  = [c for c in props if "ebull" in c.lower()][0]
col_sol  = [c for c in props if "solub" in c.lower()][0]

print(df.to_string())

# ---- 4. Hojas "Reacciones" y "Obtenciones" ----
rx = pd.read_excel(archivo, sheet_name="Reacciones").dropna(subset=["Reactante"])
ob = pd.read_excel(archivo, sheet_name="Obtenciones").dropna(subset=["Producto"])
for tabla in (rx, ob):                                  # espacios raros en los títulos -> uno solo
    tabla.columns = [re.sub(r"\s+", " ", c).strip() for c in tabla.columns]


## Celda 3 · Funciones auxiliares

**Objetivo:** reunir dos funciones pequeñas que usan varias gráficas.

- `etiquetas_x(nombres)`: parte los nombres largos (por ejemplo «Tetracloruro de carbono») en varias líneas para que quepan bajo cada punto del eje X.
- `limites_ejes(min, max, DIV)`: calcula límites y un paso «bonito» (1, 2, 2.5, 3, 4 o 5 por una potencia de 10) para que un eje tenga siempre el mismo número de divisiones. Se usa en las líneas dobles para que las líneas horizontales de la cuadrícula sirvan para leer **los dos ejes a la vez**.

In [ ]:
# ============================================================
# CELDA 3 · FUNCIONES AUXILIARES (las usan varias gráficas)
# ============================================================

def etiquetas_x(nombres):
    """Parte los nombres largos en varias líneas para que quepan en el eje X."""
    return ["\n".join(textwrap.wrap(n, 13, break_long_words=False)) for n in nombres]

def limites_ejes(minv, maxv, DIV=6):
    """Límites y paso 'bonitos': el eje arranca en 0 si no hay negativos, con DIV divisiones iguales."""
    rng = (maxv - minv) or abs(maxv) or 1
    pasos = sorted(b * 10.0 ** e for e in range(-3, 4) for b in (1, 2, 2.5, 3, 4, 5))
    for paso in pasos:
        lo = 0.0 if minv >= 0 else np.floor((minv - 0.08 * rng) / paso) * paso
        if lo + DIV * paso >= maxv + 0.08 * (maxv - lo):
            return lo, paso
    return 0.0, pasos[-1]


## Celda 4 · Líneas dobles

**Proceso**
1. Tomar del `df` el nombre del compuesto y las dos propiedades; con `solo_completos=True` se usan solo los compuestos que tienen dato en las dos.
2. **Ordenar los compuestos** de menor a mayor según la propiedad del eje izquierdo.
3. Dibujar la primera propiedad en **azul** (eje izquierdo) y la segunda en **naranja** (eje derecho, creado con `twinx`).
4. Calcular los límites de cada eje con `limites_ejes`, con 6 divisiones iguales en ambos.
5. Escribir el valor sobre cada punto: el de la línea que va más arriba se escribe arriba y el otro abajo, para que los números no se encimen.
6. Colorear cada eje del color de su línea y agregar la leyenda.

**Cómo leer la gráfica:** cada punto es un compuesto, la línea azul se lee con el eje izquierdo y la naranja con el derecho.

**¿Qué se observa?**
- *Peso molecular + densidad:* ambas tienden a subir juntas. El Fluoroetano es el menos pesado y el menos denso, el Tetracloruro de carbono es el más denso.
- *Punto de ebullición + punto de fusión:* se grafican 9 compuestos, porque al Cloruro de formilo le falta el punto de fusión. Ambas propiedades tienden a subir juntaa. El Fluoroetano tiene los valores más bajos y el Bromociclohexano el punto de ebullición más alto.

In [ ]:
# ============================================================
# CELDA 4 · LÍNEAS DOBLES (dos propiedades, dos ejes Y)
# ============================================================

def linea_doble(p, q, solo_completos=False):
    """Dos líneas con dos ejes: p (azul, izquierda) y q (naranja, derecha)."""
    datos = df[[nombre, p, q]]
    datos = datos.dropna() if solo_completos else datos.dropna(subset=[p])    # sin NA
    datos = datos.sort_values(p)
    x = list(range(len(datos)))
    nombre_p, nombre_q = p.split(" (")[0], q.split(" (")[0]
    unidad_p = p if "(" in p else p + " (g/mol)" if "peso" in p.lower() else p
    fig, ax = plt.subplots(figsize=(12, 6))
    ax2 = ax.twinx()                                      # segundo eje Y a la derecha

    l1, = ax.plot(x, datos[p], color=AZUL, marker="o", linewidth=2.2, markersize=7,
                  label=f"{nombre_p} (eje izquierdo)")
    l2, = ax2.plot(x, datos[q], color=NARANJA, marker="o", linewidth=2.2, markersize=7,
                   label=f"{nombre_q} (eje derecho)")

    # ambos ejes tienen 6 divisiones iguales, así las líneas horizontales sirven para leer los dos ejes
    DIV = 6
    lo1, s1 = limites_ejes(datos[p].min(), datos[p].max(), DIV)
    lo2, s2 = limites_ejes(datos[q].min(), datos[q].max(), DIV)
    ax.set_ylim(lo1, lo1 + DIV * s1); ax2.set_ylim(lo2, lo2 + DIV * s2)
    ax.set_yticks([lo1 + s1 * i for i in range(DIV + 1)])
    ax2.set_yticks([lo2 + s2 * i for i in range(DIV + 1)])
    ax.yaxis.grid(True, color="#e3e3e3", linewidth=0.9)      # líneas horizontales suaves
    ax.set_axisbelow(True)

    # números: el de la línea que va más arriba se escribe arriba y el otro abajo, para que no se encimen
    for xi, v1, v2 in zip(x, datos[p], datos[q]):
        pos1 = (v1 - lo1) / (DIV * s1)
        pos2 = (v2 - lo2) / (DIV * s2) if not np.isnan(v2) else -1
        arriba1 = pos1 >= pos2
        ax.annotate(f"{v1:.4g}", (xi, v1), textcoords="offset points", xytext=(0, 9 if arriba1 else -16),
                    ha="center", fontsize=9, color="#222222")
        if not np.isnan(v2):                                  # sin NA: solo donde hay dato
            ax2.annotate(f"{v2:.4g}", (xi, v2), textcoords="offset points", xytext=(0, -16 if arriba1 else 9),
                         ha="center", fontsize=9, color="#222222")

    ax.set_title(f"Comparación de {nombre_p} y {nombre_q}", fontsize=13, pad=12)
    ax.set_xticks(x); ax.set_xticklabels(etiquetas_x(datos[nombre]), fontsize=8)
    ax.set_ylabel(unidad_p, color=AZUL, fontsize=10); ax2.set_ylabel(q, color=NARANJA, fontsize=10)

    # estilo de ejes: izquierdo azul, derecho naranja, sin marco superior
    ax.tick_params(axis="y", colors=AZUL); ax2.tick_params(axis="y", colors=NARANJA)
    ax.spines["left"].set_color(AZUL); ax2.spines["right"].set_color(NARANJA)
    for e in (ax, ax2):
        e.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False); ax2.spines["left"].set_visible(False)

    fig.legend(handles=[l1, l2], loc="lower center", ncol=2, frameon=False, fontsize=10)
    fig.tight_layout(rect=[0, 0.07, 1, 1])
    plt.show()

linea_doble(col_peso, col_dens)                      # peso molecular + densidad
linea_doble(col_ebu, col_fus, solo_completos=True)   # punto de ebullición + punto de fusión


## Celda 5 · Gráfica de cajas: solubilidad en agua


**Proceso**
1. Tomar los valores de solubilidad que no están vacíos.
2. Calcular con `np.percentile` el **mínimo, Q1 (25 %), mediana (50 %), Q3 (75 %) y máximo**.
3. Dibujar la caja desde Q1 hasta Q3, dividida en dos colores (rosa y verde) en la mediana, con «bigotes» hasta el mínimo y el máximo.

**Cómo leer la gráfica:** la caja contiene el 50 % central de los datos; el cambio de color marca la mediana.

**Qué se observa:** solo 4 compuestos tienen dato (n = 4). Mínimo 5.33 g/L, Q1 7.67, mediana 9.225, Q3 12.5 y máximo 20 g/L. Con tan pocos datos la distribución es solo **referencial**.

In [ ]:
# ============================================================
# CELDA 5 · GRÁFICA DE CAJAS: SOLUBILIDAD EN AGUA
# ============================================================

def caja_solubilidad():
    """Gráfica de cajas horizontal: la caja se divide en dos colores en la mediana."""
    v = df[col_sol].dropna().to_numpy()
    q1, me, q3 = np.percentile(v, [25, 50, 75])         # cuartiles y mediana
    mn, mx = v.min(), v.max()
    MOSTRAR_VALORES = False           # pon True para mostrar los números sobre la caja
    alto = 0.36

    fig, ax = plt.subplots(figsize=(11, 4.4))
    ax.axhline(0, color="#b5b5b5", linestyle=(0, (4, 4)), linewidth=1, zorder=0)         # línea punteada de la fila
    ax.add_patch(Rectangle((q1, -alto / 2), me - q1, alto, facecolor=ROSA, edgecolor="black", linewidth=1, zorder=2))
    ax.add_patch(Rectangle((me, -alto / 2), q3 - me, alto, facecolor=VERDE, edgecolor="black", linewidth=1, zorder=2))
    ax.plot([mn, q1], [0, 0], color="black", linewidth=1.8, zorder=1)                    # bigote izquierdo
    ax.plot([q3, mx], [0, 0], color="black", linewidth=1.8, zorder=1)                    # bigote derecho
    for borde in (mn, mx):
        ax.plot([borde, borde], [-0.1, 0.1], color="black", linewidth=1.8)               # topes de los bigotes

    if MOSTRAR_VALORES:
        for etiqueta, xv in (("Mín", mn), ("Q₁", q1), ("Me", me), ("Q₃", q3), ("Máx", mx)):
            ax.text(xv, alto / 2 + 0.06, f"{etiqueta}\n{xv:.4g}", ha="center", va="bottom",
                    fontsize=9, color="#333333")

    ax.set_xlim(0, mx * 1.1); ax.set_ylim(-0.7, 0.75)
    ax.set_yticks([0]); ax.set_yticklabels([col_sol.split(" (")[0]], fontsize=10)
    ax.tick_params(axis="y", length=0)
    ax.xaxis.grid(True, color="#bdbdbd", linewidth=0.9)                                  # líneas verticales
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.set_xlabel(col_sol)
    ax.set_title("Distribución de " + col_sol.split(" (")[0], fontsize=13, pad=12)
    plt.tight_layout()
    plt.show()

caja_solubilidad()


## Celda 6 · Violín: Reacciones vs Obtenciones


**Proceso**
1. `pka_celda` y `valores_pka` limpian el pKa de cada celda: arreglan valores que Excel convirtió en fecha (por ejemplo 15.5 → «15 de mayo»), decimales perdidos (16374 → 16.374) y celdas con varios valores (se usa el menor, el más ácido).
2. `log_presion` convierte la presión de vapor a **log₁₀**, porque sus valores abarcan muchísimos órdenes de magnitud.
3. `leer_columna` aplica esas funciones a las hojas `rx` y `ob` y deja solo los valores válidos.
4. `dibujar_violin` dibuja un violín por grupo (el ancho muestra cuántos datos hay en cada valor), con una barra gruesa para el rango intercuartil (Q1–Q3), un punto blanco para la mediana y una línea fina de mínimo a máximo.

**Cómo leer la gráfica:** cuanto más ancho es el violín en un valor, más registros hay ahí; el punto blanco es la mediana de cada grupo.

**Qué se observa**
- *pKa:* mediana de 5.1 en Reacciones (n = 34) y de 13.82 en Obtenciones (n = 46). Ambos grupos tienen rangos muy amplios.
- *Presión de vapor (log₁₀ Pa):* mediana de −3.66 en Reacciones (n = 44) y de 5.01 en Obtenciones (n = 51); es decir, las sustancias de las Obtenciones tienen presiones de vapor mucho mayores en conjunto.

In [ ]:
# ============================================================
# CELDA 6 · VIOLÍN: REACCIONES vs OBTENCIONES (pKa y presión de vapor)
# ============================================================

def valores_pka(celda):
    """Valores de pKa de una celda del Excel (arregla fechas, decimales perdidos y celdas con varios valores)."""
    if isinstance(celda, (datetime.datetime, pd.Timestamp)):
        return [celda.day + celda.month / 10]              # '15.5' que Excel convirtió en la fecha 15 de mayo
    salida = []
    for parte in re.split(r"[\n;]+", str(celda)):
        v = a_numero(parte)
        if not np.isnan(v):
            salida.append(v / 1000 if abs(v) >= 100 else v)  # decimal perdido: 16374 -> 16.374
    return salida

def pka_celda(x):                                          # si la celda trae varios, se usa el más ácido (el menor)
    v = valores_pka(x)
    return min(v) if v else None

def log_presion(x):                                        # la presión de vapor se grafica en log10
    v = a_numero(x)
    return np.log10(v) if (not np.isnan(v) and v > 0) else None

def leer_columna(tabla, clave_col, funcion):
    """Aplica 'funcion' a cada celda de la columna que contiene 'clave_col' y deja solo los valores válidos."""
    col = [c for c in tabla.columns if clave_col in c.lower()][0]
    return np.array([v for v in (funcion(x) for x in tabla[col]) if v is not None and not np.isnan(v)], dtype=float)

def dibujar_violin(ax, datos_rx, datos_ob, titulo, etiqueta_y):
    """Dibuja un violín azul (Reacciones) y uno naranja (Obtenciones) en el eje 'ax'."""
    for pos, datos, color in ((1, datos_rx, AZUL), (2, datos_ob, NARANJA)):
        cuerpo = ax.violinplot(datos, positions=[pos], widths=0.8, showextrema=False)["bodies"][0]
        cuerpo.set_facecolor(color); cuerpo.set_edgecolor(color); cuerpo.set_alpha(0.45); cuerpo.set_linewidth(1.4)
        q1, me, q3 = np.percentile(datos, [25, 50, 75])
        ax.vlines(pos, datos.min(), datos.max(), color=color, linewidth=1.6, zorder=3)          # mínimo y máximo
        ax.vlines(pos, q1, q3, color=color, linewidth=8, zorder=4)                              # Q1 - Q3
        ax.plot(pos, me, "o", color="white", markeredgecolor=color, markersize=7, zorder=5)     # mediana
    ax.set_title(titulo, fontsize=13, pad=12)
    ax.set_ylabel(etiqueta_y, fontsize=10)
    ax.set_xticks([1, 2])
    ax.set_xticklabels([f"Reacciones\n(n = {len(datos_rx)})", f"Obtenciones\n(n = {len(datos_ob)})"], fontsize=9)
    ax.set_xlim(0.4, 2.6)
    ax.yaxis.grid(True, color="#e3e3e3", linewidth=0.9)       # líneas horizontales suaves
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

def violines_reacciones_vs_obtenciones():
    pka_rx = leer_columna(rx, "pka", pka_celda)
    pka_ob = leer_columna(ob, "pka", pka_celda)
    pv_rx = leer_columna(rx, "presi", log_presion)
    pv_ob = leer_columna(ob, "presi", log_presion)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6.4))
    dibujar_violin(ax1, pka_rx, pka_ob, "pKa: Reacciones vs Obtenciones", "pKa")
    dibujar_violin(ax2, pv_rx, pv_ob, "Presión de vapor: Reacciones vs Obtenciones", "log₁₀ Presión de vapor (Pa)")

    fig.legend(handles=[Patch(facecolor=AZUL, alpha=0.6, label="Reacciones"),
                        Patch(facecolor=NARANJA, alpha=0.6, label="Obtenciones")],
               loc="lower center", bbox_to_anchor=(0.5, 0.05), ncol=2, frameon=False, fontsize=10)
    fig.text(0.5, 0.012, "Barra gruesa = rango intercuartil (Q1-Q3) · punto blanco = mediana · línea fina = mínimo y máximo",
             ha="center", fontsize=8.5, color="#555555")
    fig.subplots_adjust(left=0.07, right=0.98, top=0.88, bottom=0.2, wspace=0.18)
    plt.show()

violines_reacciones_vs_obtenciones()


## Celda 7 · Mapa de calor de correlación entre propiedades


**Proceso** (función `mapa_calor_correlacion`)
1. Elegir 6 propiedades: solubilidad, peso molecular, momento dipolar, punto de ebullición, punto de fusión y densidad. El pKa se deja fuera por tener un solo dato.
2. Calcular la **correlación de Pearson** entre cada par (`df.corr`), ignorando los datos vacíos y exigiendo al menos 3 pares de datos (`min_periods=3`). La diagonal vale 1.
3. **Ordenar** filas y columnas con agrupamiento jerárquico (`linkage` y `leaves_list`), para que las propiedades parecidas queden juntas.
4. Dibujar el mapa con degradado turquesa y escribir el valor en cada cuadro; las celdas sin dato quedan en gris.
5. Agregar al lado una tabla que explica cada símbolo.

**Cómo leer la gráfica:** cerca de **1** las dos propiedades suben juntas; cerca de **0** no hay relación lineal; cerca de **−1** una sube cuando la otra baja.

**Qué se observa:** la relación más fuerte es **peso molecular – punto de ebullición (0.88)**. Siguen peso molecular – densidad (0.73), peso molecular – punto de fusión (0.72) y punto de ebullición – densidad (0.69). La solubilidad casi no se relaciona con el peso molecular (0.05). Al haber pocos compuestos con datos, estas correlaciones son orientativas.

In [ ]:
# ============================================================
# CELDA 7 · MAPA DE CALOR DE CORRELACIÓN ENTRE PROPIEDADES
# ============================================================

MOSTRAR_NUMEROS = True        # pon False para dejar los cuadros sin números

# Símbolo de cada propiedad (puedes cambiarlos aquí)
SIMBOLOS = {
    "peso":     "PM",
    "fusi":     "Pf",
    "ebull":    "Peb",
    "densidad": "ρ",     # ρ
    "momento":  "μ",     # μ
    "solub":    "S",
    "presi":    "Pv",
}
def simbolo(p):
    for clave, sim in SIMBOLOS.items():
        if clave in p.lower():
            return sim
    return p

# Propiedades que se correlacionan (sin pKa, como en el ejemplo)
CLAVES = ["solub", "peso", "momento", "ebull", "fusi", "densidad"]

def mapa_calor_correlacion():
    cols = [[c for c in props if k in c.lower()][0] for k in CLAVES]

    corr = df[cols].corr(min_periods=3)                # correlación entre propiedades (se ignoran los vacíos)
    for c in corr.index:
        corr.loc[c, c] = 1.0                           # la diagonal siempre vale 1.00

    # Orden por agrupamiento: igual en filas y columnas, así la línea de unos queda en la diagonal
    Z = linkage(corr.fillna(0).values, method="average", metric="euclidean")
    orden = leaves_list(Z)
    corr = corr.iloc[orden, orden]
    simbolos = [simbolo(c) for c in corr.columns]

    # Datos de la tabla de símbolos: [símbolo, nombre, unidad]
    filas = []
    for p in corr.columns:
        unidad = re.search(r"\((.*?)\)", p)
        nombre_corto = re.sub(r"\s*\(.*?\)", "", p)
        u = unidad.group(1) if unidad else ("g/mol" if "peso" in p.lower() else "—")
        filas.append([simbolo(p), nombre_corto, u])

    # Figura: mapa de calor a la izquierda, tabla a la derecha
    fig, (ax, axt) = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [3, 2.4], "wspace": 0.3})
    im = ax.imshow(corr.values, cmap=turquesa, vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(simbolos))); ax.set_xticklabels(simbolos, fontsize=12)
    ax.set_yticks(range(len(simbolos))); ax.set_yticklabels(simbolos, fontsize=12)

    if MOSTRAR_NUMEROS:                                # números dentro de cada cuadrito (vacíos: nada)
        for i in range(corr.shape[0]):
            for j in range(corr.shape[1]):
                v = corr.iat[i, j]
                if np.isnan(v):
                    continue
                color_texto = "white" if (v + 1) / 2 > 0.55 else "#1a1a1a"
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10, color=color_texto)
    ax.tick_params(length=0)
    for borde in ax.spines.values():
        borde.set_visible(False)
    fig.colorbar(im, ax=ax, label="Correlación (-1 a 1)", fraction=0.04, pad=0.02, ticks=[-1, -0.5, 0, 0.5, 1])
    ax.set_title("Mapa de calor de correlación entre propiedades", fontsize=16, pad=18)

    # Tabla en forma de cuadro redondeado (turquesa claro, letra negra)
    axt.axis("off")
    axt.set_xlim(0, 1); axt.set_ylim(0, 1)
    n = len(filas)
    alto = 0.15 + 0.07 * n
    y_sup = 0.97
    y_inf = y_sup - alto

    caja = FancyBboxPatch((0.04, y_inf), 0.92, alto,
                          boxstyle="round,pad=0.0,rounding_size=0.05",
                          mutation_aspect=0.45,
                          facecolor=to_hex(turquesa(0.2)), edgecolor=to_hex(turquesa(0.5)), linewidth=3,
                          transform=axt.transData, clip_on=False)
    axt.add_patch(caja)

    axt.text(0.5, y_sup - 0.06, "PROPIEDADES FISICOQUÍMICAS", ha="center", va="center",
             fontsize=13, fontweight="bold", color="black")
    axt.plot([0.18, 0.82], [y_sup - 0.09, y_sup - 0.09], color="black", linewidth=1)   # subrayado

    for k, (sim, nom, uni) in enumerate(filas):
        y = y_sup - 0.15 - 0.07 * k
        unidad_txt = f" ({uni})" if uni != "—" else ""
        axt.text(0.10, y, f"{sim} = {nom.upper()}{unidad_txt}", ha="left", va="center",
                 fontsize=10.5, fontweight="bold", color="black")

    fig.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.12)
    plt.show()

mapa_calor_correlacion()


## Celda 8 · Barras: tipos de reacción

**Proceso** (función `barras_tipos`)
1. Contar con `value_counts` cuántos registros hay de cada tipo, de mayor a menor.
2. Dibujar barras horizontales: la más frecuente en **rojo** y las demás en azul claro.
3. Escribir al lado de cada barra solo el **porcentaje** que representa del total.

**Cómo leer la gráfica:** la barra más larga (en rojo) es el tipo de reacción más frecuente.

**Qué se observa:** en ambas hojas predomina la **sustitución nucleofílica**: 44 % de las Reacciones y 52 % de las Obtenciones. Después, en Reacciones siguen condensación (9 %) y sustitución radicalaria (8 %); en Obtenciones, oxidación (14 %) y eliminación (9 %).

In [ ]:
# ============================================================
# CELDA 8 · BARRAS: TIPOS DE REACCIÓN (el más frecuente en rojo)
# ============================================================

def barras_tipos(tabla, titulo):
    t = tabla["Tipo de la reacción"].value_counts()          # de mayor a menor
    total = t.sum()
    colores = [ROJO if v == t.max() else AZUL_CLARO for v in t.values]

    fig, ax = plt.subplots(figsize=(11, 0.45 * len(t) + 2))
    barras = ax.barh(t.index, t.values, color=colores, edgecolor="none", height=0.65)
    ax.invert_yaxis()                                         # el más alto arriba
    for barra, v in zip(barras, t.values):                    # solo el porcentaje al lado de cada barra
        ax.text(v + t.max() * 0.012, barra.get_y() + barra.get_height() / 2,
                f"{v / total * 100:.0f}%", va="center", fontsize=9, color="#222222")
    ax.set_xlim(0, t.max() * 1.15)
    ax.set_xlabel("Número de registros")
    ax.set_title(f"Tipos de reacción - {titulo}", fontsize=13, pad=12)
    ax.xaxis.grid(True, color="#e3e3e3", linewidth=0.9)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    plt.tight_layout()
    plt.show()

barras_tipos(rx, "Reacciones")
barras_tipos(ob, "Obtenciones")


## Celda 9 · Lollipop: comparación con el Cloroformo


**Proceso**
1. Ordenar los compuestos de menor a mayor valor y tomar el valor del compuesto de referencia.
2. Dibujar una línea punteada horizontal en el valor de la referencia.
3. Para cada compuesto, dibujar un «palito» desde la línea de referencia hasta su valor, terminado en un punto: **naranja** si es mayor que la referencia, **azul** si es menor, y gris oscuro para el Cloroformo.
4. Escribir el valor y, entre paréntesis, la **diferencia** con la referencia.

**Cómo leer la gráfica:** cuanto más largo es el palito, más se aleja ese compuesto del Cloroformo; hacia arriba tiene un valor mayor y hacia abajo uno menor.

**Qué se observa**
- *Punto de ebullición* (Cloroformo: 61.2 °C): solo tres compuestos hierven más alto, Bromuro de acetilo (+14.8), Tetracloruro de carbono (+15.6) y Bromociclohexano (+104.8). Los otros seis hierven más bajo, hasta el Fluoroetano (−98.8).
- *Punto de fusión* (Cloroformo: −63.2 °C): funden más alto el Bromociclohexano (+6.7) y el Tetracloruro de carbono (+40.2); los otros seis funden más bajo, hasta el Fluoroetano (−80.0). Son 9 compuestos, porque falta el dato del Cloruro de formilo.

In [ ]:
# ============================================================
# CELDA 9 · LOLLIPOP: TEMPERATURAS COMPARADAS CON UN COMPUESTO DE REFERENCIA
# ============================================================

def lollipop_temperatura(col, ref="Cloroformo", mas_alto="Más alto", mas_bajo="Más bajo"):
    """Lollipop: cada palito sale de la línea del compuesto de referencia hasta el valor del compuesto."""
    nombre_prop = col.split(" (")[0]
    datos = df[[nombre, col]].dropna().sort_values(col).reset_index(drop=True)
    val_ref = float(datos.loc[datos[nombre] == ref, col].iloc[0])
    x = np.arange(len(datos))
    fig, ax = plt.subplots(figsize=(12, 6.5))

    ax.axhline(val_ref, color="#444444", linestyle=":", linewidth=1.6, zorder=1)            # línea de referencia
    ax.text(-0.55, val_ref, f"{ref}: {val_ref:g} °C", ha="left", va="bottom",
            fontsize=10, fontweight="bold", color="#444444")

    for xi, n, v in zip(x, datos[nombre], datos[col]):
        es_ref = n == ref
        color = "#444444" if es_ref else (NARANJA if v > val_ref else AZUL)
        ax.vlines(xi, val_ref, v, color=color, linewidth=2.4, zorder=2)                     # palito
        ax.scatter(xi, v, s=(260 if es_ref else 170), color=color, edgecolor="white", linewidth=2, zorder=3)
        if es_ref:
            txt = f"{v:.4g} °C"
            arriba = True
        else:
            arriba = v > val_ref
            txt = f"{v:.4g} °C\n({v - val_ref:+.1f})"                                      # valor y diferencia
        ax.annotate(txt, (xi, v), textcoords="offset points", xytext=(0, 14 if arriba else -14),
                    ha="center", va="bottom" if arriba else "top", fontsize=9.5, fontweight="bold", color=color)

    ax.set_xticks(x); ax.set_xticklabels(etiquetas_x(datos[nombre]), fontsize=9)
    for lbl in ax.get_xticklabels():
        if lbl.get_text().replace("\n", " ") == ref:
            lbl.set_fontweight("bold")
    ax.tick_params(axis="x", length=0, pad=8)
    ax.set_xlim(-0.6, len(datos) - 0.4)
    ax.margins(y=0.18)
    ax.set_ylabel(col, fontsize=10.5)
    ax.set_title(f"{nombre_prop} de cada compuesto comparado con {ref} ({val_ref:g} °C)",
                 fontsize=13.5, fontweight="bold", pad=14)
    ax.yaxis.grid(True, color="#e8e8e8", linewidth=0.9); ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

    ax.legend(handles=[Line2D([], [], marker="o", color="w", markerfacecolor=NARANJA, markersize=11, label=f"{mas_alto} que {ref}"),
                       Line2D([], [], marker="o", color="w", markerfacecolor=AZUL, markersize=11, label=f"{mas_bajo} que {ref}")],
              loc="upper left", frameon=False, fontsize=10)
    fig.tight_layout()
    plt.show()

lollipop_temperatura(col_ebu, "Cloroformo", "Hierve más alto", "Hierve más bajo")                        # punto de ebullición
lollipop_temperatura(col_fus, "Cloroformo", "Funde a mayor temperatura", "Funde a menor temperatura")   # punto de fusión


## Celda 10 · Tortas: medio de reacción

**Proceso**
1. `conteo_medio` cuenta cuántos registros hay de cada medio, unificando los valores escritos en dos líneas (`Ácido\nBásico` → `Ácido/Básico`).
2. `torta_medio` dibuja una torta por hoja, con el **porcentaje** en cada porción y los colores crema/beige definidos en la Celda 1.
3. `tortas_medio` arma la figura con las dos tortas y agrega al lado una **leyenda** con el color y el significado de cada medio.

**Cómo leer la gráfica:** el tamaño de cada porción es el porcentaje de reacciones en ese medio; el color se identifica con la leyenda de la derecha.

**Qué se observa**
- *Reacciones:* Neutro 34 %, Básico 32 %, Ácido 30 % y Ácido/Básico 4 %; están repartidas casi por igual entre los tres medios principales.
- *Obtenciones:* predomina el medio Ácido (61 %), seguido de Neutro (32 %) y Básico (7 %).

In [ ]:
# ============================================================
# CELDA 10 · TORTAS: MEDIO DE REACCIÓN (con leyenda al lado)
# ============================================================

DESCRIPCION = {
    "Ácido":        "Reacción en medio ácido",
    "Básico":       "Reacción en medio básico",
    "Neutro":       "Reacción en medio neutro",
    "Ácido/Básico": "Reacción que ocurre en medio ácido y básico",
}

def conteo_medio(tabla):
    medio = (tabla["Medio"].astype(str)
             .str.replace(r"\s*\n\s*", "/", regex=True).str.strip())   # "Ácido\nBásico" -> "Ácido/Básico"
    return medio.value_counts()

def torta_medio(tabla, titulo, ax):
    c = conteo_medio(tabla)
    ax.set_facecolor(FONDO)
    ax.pie(c.values, colors=[COLORES_MEDIO.get(k, "#e6e6e6") for k in c.index],
           autopct=lambda p: f"{p:.0f}%", startangle=90, counterclock=False,
           pctdistance=0.68, wedgeprops={"edgecolor": FONDO, "linewidth": 2.5},
           textprops={"fontsize": 12, "color": "#333333", "fontweight": "bold"})
    ax.set_title(f"Medio de reacción - {titulo}\n(n = {c.sum()})", fontsize=13, pad=12, color="#333333")
    ax.axis("equal")

def tortas_medio():
    fig = plt.figure(figsize=(17, 6.5), facecolor=FONDO)
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.8], left=0.02, right=0.98, top=0.88, bottom=0.06, wspace=0.08)
    ax1, ax2, axl = (fig.add_subplot(gs[i]) for i in range(3))
    torta_medio(rx, "Reacciones", ax1)
    torta_medio(ob, "Obtenciones", ax2)

    # Leyenda al lado: cuadro redondeado con el color y su significado
    presentes = [k for k in COLORES_MEDIO if k in set(conteo_medio(rx).index) | set(conteo_medio(ob).index)]
    axl.axis("off"); axl.set_xlim(0, 1); axl.set_ylim(0, 1); axl.set_facecolor(FONDO)
    alto = 0.16 + 0.13 * len(presentes)
    y_sup = 0.5 + alto / 2; y_inf = y_sup - alto
    axl.add_patch(FancyBboxPatch((0.04, y_inf), 0.92, alto, boxstyle="round,pad=0.0,rounding_size=0.04",
                  mutation_aspect=1.8, facecolor="white", edgecolor="#cfb384", linewidth=2.5, clip_on=False))
    axl.text(0.5, y_sup - 0.06, "MEDIO DE REACCIÓN", ha="center", va="center", fontsize=12, fontweight="bold", color="#333333")
    axl.plot([0.15, 0.85], [y_sup - 0.095, y_sup - 0.095], color="#333333", linewidth=1)
    for k, nombre_m in enumerate(presentes):
        y = y_sup - 0.19 - 0.13 * k
        axl.add_patch(Rectangle((0.10, y - 0.035), 0.09, 0.07, facecolor=COLORES_MEDIO[nombre_m], edgecolor="#8a8a8a", linewidth=0.8))
        axl.text(0.23, y + 0.018, nombre_m, ha="left", va="center", fontsize=11, fontweight="bold", color="#222222")
        axl.text(0.23, y - 0.022, DESCRIPCION[nombre_m], ha="left", va="center", fontsize=8.5, color="#444444")
    plt.show()

tortas_medio()
